# Fine-tuning Qwen2.5-1.5B với QLoRA cho Suy luận Pháp lý Tiếng Việt

**Hướng dẫn sử dụng:**
1. Đảm bảo Runtime → Change runtime type → **T4 GPU**
2. Chạy từng cell theo thứ tự
3. Dataset tự động tải từ HuggingFace (`VLSP2025-LegalSML/Public-Test`)
4. Copy kết quả từ Cell 8 vào báo cáo

**Thời gian ước tính:** ~45-60 phút (cài đặt + train + đánh giá)

## Cell 1: Kiểm tra GPU và cài đặt thư viện

In [ ]:
# Kiểm tra GPU
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# Cài đặt thư viện - phiên bản tương thích Colab T4
!pip install -q transformers peft trl accelerate datasets rouge-score
!pip install -q --upgrade bitsandbytes
print("Cài đặt hoàn tất! Hãy Restart Runtime trước khi chạy các cell tiếp theo.")
print("Menu: Runtime → Restart session")

## Cell 2: Tải dữ liệu từ HuggingFace Hub

In [ ]:
# Đăng nhập HuggingFace nếu dataset yêu cầu (bỏ comment nếu cần)
# from huggingface_hub import login
# login(token="hf_xxx")  # thay bằng token của bạn

from datasets import load_dataset
import random
from sklearn.model_selection import train_test_split

random.seed(42)

print("Đang tải dataset từ HuggingFace...")
ds_mcq = load_dataset("VLSP2025-LegalSML/Public-Test", "multichoice_questions")
ds_nli = load_dataset("VLSP2025-LegalSML/Public-Test", "nli_questions")
ds_syl = load_dataset("VLSP2025-LegalSML/Public-Test", "syllogism_questions")

print("Cấu trúc dataset:")
print(f"  MCQ splits: {list(ds_mcq.keys())}")
print(f"  NLI splits: {list(ds_nli.keys())}")
print(f"  Syllogism splits: {list(ds_syl.keys())}")

# Lấy split có dữ liệu (train / test / validation tùy dataset)
def get_split(ds):
    for split in ['train', 'test', 'validation']:
        if split in ds:
            return list(ds[split])
    return list(ds[list(ds.keys())[0]])

mcq_data = get_split(ds_mcq)
nli_data  = get_split(ds_nli)
syl_data  = get_split(ds_syl)

print(f"\nSố mẫu tải được:")
print(f"  MCQ:      {len(mcq_data)} mẫu | Fields: {list(mcq_data[0].keys())}")
print(f"  NLI:      {len(nli_data)} mẫu  | Fields: {list(nli_data[0].keys())}")
print(f"  Syllogism:{len(syl_data)} mẫu  | Fields: {list(syl_data[0].keys())}")
print(f"  Tổng: {len(mcq_data)+len(nli_data)+len(syl_data)} mẫu")
print("\nVí dụ 1 mẫu MCQ:")
print(mcq_data[0])

## Cell 3: Định dạng dữ liệu theo Instruction Tuning

In [ ]:
SYSTEM_PROMPT = """Bạn là một chuyên gia pháp lý Việt Nam. Nhiệm vụ của bạn là phân tích văn bản pháp luật và trả lời các câu hỏi pháp lý một cách chính xác, súc tích."""

def format_nli(sample):
    """Format NLI sample thành instruction tuning"""
    choices_text = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'])])
    user_msg = f"""Văn bản pháp lý:
{sample['legal_document']}

Câu hỏi cụ thể: {sample['specific_question']}

{sample['question']}
{choices_text}

Hãy chọn đáp án đúng (chỉ trả lời bằng chữ cái A hoặc B)."""
    
    answer_letter = chr(65 + sample['answer'])
    return {"system": SYSTEM_PROMPT, "user": user_msg, "assistant": answer_letter, "task": "nli"}

def format_mcq(sample):
    """Format MCQ sample thành instruction tuning"""
    choices_text = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'])])
    user_msg = f"""Câu hỏi pháp luật: {sample['question']}

{choices_text}

Hãy chọn đáp án đúng (chỉ trả lời bằng chữ cái A, B, C hoặc D)."""
    
    answer_letter = chr(65 + sample['answer'])
    return {"system": SYSTEM_PROMPT, "user": user_msg, "assistant": answer_letter, "task": "mcq"}

def format_syllogism(sample):
    """Format Syllogism sample thành instruction tuning"""
    user_msg = f"""Tình huống pháp lý:
{sample['question']}

Hãy phân tích tình huống trên theo các bước:
1. Xác định điều khoản pháp luật áp dụng (Tiền đề lớn)
2. Áp dụng vào tình huống cụ thể (Tiền đề nhỏ)
3. Đưa ra kết luận"""
    
    return {"system": SYSTEM_PROMPT, "user": user_msg, "assistant": sample['answer'], "task": "syllogism"}

# Chuyển đổi dữ liệu
nli_formatted = [format_nli(s) for s in nli_data]
mcq_formatted = [format_mcq(s) for s in mcq_data]
syl_formatted = [format_syllogism(s) for s in syl_data]

print("Ví dụ NLI sau định dạng:")
print(f"  User: {nli_formatted[0]['user'][:200]}...")
print(f"  Assistant: {nli_formatted[0]['assistant']}")
print()
print("Ví dụ MCQ sau định dạng:")
print(f"  User: {mcq_formatted[0]['user'][:200]}...")
print(f"  Assistant: {mcq_formatted[0]['assistant']}")

In [ ]:
# Phân chia train/test (80/20)
random.seed(42)

nli_train, nli_test = train_test_split(nli_formatted, test_size=0.2, random_state=42)
mcq_train, mcq_test = train_test_split(mcq_formatted, test_size=0.2, random_state=42)
syl_train, syl_test = train_test_split(syl_formatted, test_size=0.2, random_state=42)

# Gộp train
train_data = nli_train + mcq_train + syl_train
random.shuffle(train_data)

print(f"Train: NLI={len(nli_train)}, MCQ={len(mcq_train)}, Syllogism={len(syl_train)}, Tổng={len(train_data)}")
print(f"Test:  NLI={len(nli_test)},  MCQ={len(mcq_test)},  Syllogism={len(syl_test)},  Tổng={len(nli_test)+len(mcq_test)+len(syl_test)}")

## Cell 4: Tải mô hình Qwen2.5-1.5B với 4-bit quantization

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Cấu hình 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,  # Double quantization
)

print(f"Đang tải mô hình {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Mô hình đã tải xong!")
print(f"Số tham số: {model.num_parameters():,}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Thử 4-bit, fallback sang float16 nếu lỗi
try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    print(f"Đang tải mô hình {MODEL_NAME} với 4-bit quantization...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print("Tải thành công với 4-bit QLoRA!")

except Exception as e:
    print(f"4-bit thất bại ({e})\nFallback sang float16...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    print("Tải thành công với float16 (sẽ dùng LoRA thay QLoRA)!")

print(f"Số tham số: {model.num_parameters():,}")
print(f"Device: {next(model.parameters()).device}")

In [ ]:
def generate_response(model, tokenizer, system, user, max_new_tokens=256):
    """Sinh câu trả lời từ mô hình"""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return response.strip()

def extract_choice(text):
    """Trích xuất lựa chọn A/B/C/D từ câu trả lời"""
    import re
    text = text.strip().upper()
    # Tìm chữ cái đầu tiên trong A-D
    match = re.search(r'^([ABCD])', text)
    if match:
        return match.group(1)
    # Tìm trong toàn bộ text
    for ch in ['A', 'B', 'C', 'D']:
        if ch in text[:10]:
            return ch
    return text[0] if text else 'A'

def evaluate_classification(model, tokenizer, test_data, task_name, max_tokens=16):
    """Đánh giá accuracy cho NLI và MCQ"""
    correct = 0
    results = []
    
    for i, sample in enumerate(test_data):
        response = generate_response(model, tokenizer, sample['system'], sample['user'], max_new_tokens=max_tokens)
        pred = extract_choice(response)
        gold = sample['assistant']
        is_correct = (pred == gold)
        correct += is_correct
        results.append({'pred': pred, 'gold': gold, 'correct': is_correct, 'response': response})
        if (i+1) % 5 == 0:
            print(f"  [{task_name}] {i+1}/{len(test_data)} - Running acc: {correct/(i+1):.2%}")
    
    accuracy = correct / len(test_data)
    print(f"  [{task_name}] Accuracy: {accuracy:.4f} ({correct}/{len(test_data)})")
    return accuracy, results

print("=== BASELINE ZERO-SHOT ===")
print("Đánh giá NLI...")
zs_nli_acc, zs_nli_results = evaluate_classification(model, tokenizer, nli_test, "NLI-ZS")

print("Đánh giá MCQ...")
zs_mcq_acc, zs_mcq_results = evaluate_classification(model, tokenizer, mcq_test, "MCQ-ZS")

print(f"\nTóm tắt Zero-shot:")
print(f"  NLI Accuracy: {zs_nli_acc:.4f}")
print(f"  MCQ Accuracy: {zs_mcq_acc:.4f}")

In [ ]:
# Zero-shot Syllogism với ROUGE
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def evaluate_syllogism(model, tokenizer, test_data):
    """Đánh giá ROUGE-L cho Syllogism"""
    rouge_scores = []
    results = []
    
    for i, sample in enumerate(test_data):
        response = generate_response(model, tokenizer, sample['system'], sample['user'], max_new_tokens=512)
        score = scorer.score(sample['assistant'], response)['rougeL'].fmeasure
        rouge_scores.append(score)
        results.append({'pred': response, 'gold': sample['assistant'], 'rouge_l': score})
        print(f"  [Syllogism] {i+1}/{len(test_data)} - ROUGE-L: {score:.4f}")
    
    avg_rouge = sum(rouge_scores) / len(rouge_scores)
    print(f"  [Syllogism] Avg ROUGE-L: {avg_rouge:.4f}")
    return avg_rouge, results

print("Đánh giá Syllogism (zero-shot)...")
zs_syl_rouge, zs_syl_results = evaluate_syllogism(model, tokenizer, syl_test)
print(f"\nZero-shot Syllogism ROUGE-L: {zs_syl_rouge:.4f}")

## Cell 6: Cấu hình và huấn luyện QLoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import Dataset

# Chuẩn bị mô hình cho QLoRA
model = prepare_model_for_kbit_training(model)

# Cấu hình LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Chuyển train_data thành format cho SFTTrainer
def format_for_sft(sample):
    """Chuyển sample thành chuỗi huấn luyện hoàn chỉnh"""
    messages = [
        {"role": "system", "content": sample['system']},
        {"role": "user", "content": sample['user']},
        {"role": "assistant", "content": sample['assistant']}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

# Tạo dataset
train_texts = [format_for_sft(s) for s in train_data]
train_dataset = Dataset.from_dict({"text": train_texts})

print(f"Số mẫu huấn luyện: {len(train_dataset)}")
print(f"Ví dụ:\n{train_texts[0][:500]}...")

import time

# Cấu hình huấn luyện
training_args = TrainingArguments(
    output_dir="./qwen_legal_lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none",
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    packing=False,
)

print("Bắt đầu huấn luyện...")
start_time = time.time()
train_result = trainer.train()
end_time = time.time()

training_time = (end_time - start_time) / 60
print(f"\nHuấn luyện hoàn tất!")
print(f"Thời gian: {training_time:.1f} phút")
print(f"Training loss cuối: {train_result.training_loss:.4f}")

In [ ]:
import time, inspect
from trl import SFTTrainer, SFTConfig

valid_sft_config = inspect.signature(SFTConfig.__init__).parameters
valid_sft_trainer = inspect.signature(SFTTrainer.__init__).parameters

base_args = dict(
    output_dir="./qwen_legal_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    fp16=False,
    bf16=False,
    logging_steps=5,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none",
    dataloader_num_workers=0,
)
for key, val in [("max_seq_length", 1024), ("max_length", 1024),
                 ("dataset_text_field", "text"), ("packing", False)]:
    if key in valid_sft_config:
        base_args[key] = val

sft_config = SFTConfig(**base_args)

trainer_kwargs = dict(model=model, train_dataset=train_dataset, args=sft_config)
if "processing_class" in valid_sft_trainer:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in valid_sft_trainer:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

print("Bắt đầu huấn luyện...")
start_time = time.time()
train_result = trainer.train()
end_time = time.time()

training_time = (end_time - start_time) / 60
print(f"\nHuấn luyện hoàn tất!")
print(f"Thời gian: {training_time:.1f} phút")
print(f"Training loss cuối: {train_result.training_loss:.4f}")

In [ ]:
print("=== ĐÁNH GIÁ SAU FINE-TUNING (QLoRA Multi-task) ===")
model.eval()

print("Đánh giá NLI...")
ft_nli_acc, ft_nli_results = evaluate_classification(model, tokenizer, nli_test, "NLI-FT")

print("Đánh giá MCQ...")
ft_mcq_acc, ft_mcq_results = evaluate_classification(model, tokenizer, mcq_test, "MCQ-FT")

print("Đánh giá Syllogism...")
ft_syl_rouge, ft_syl_results = evaluate_syllogism(model, tokenizer, syl_test)

In [ ]:
# Load lại model + adapter sau training (chạy cell này nếu model bị mất)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "./qwen_legal_lora"  # hoặc "./qwen_legal_lora/checkpoint-xxx"

# Tìm checkpoint mới nhất nếu có
import os, glob
checkpoints = sorted(glob.glob(f"{ADAPTER_PATH}/checkpoint-*"))
adapter_path = checkpoints[-1] if checkpoints else ADAPTER_PATH
print(f"Load adapter từ: {adapter_path}")

try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
    )
except Exception as e:
    print(f"4-bit thất bại ({e}), dùng float16...")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()
print("Load thành công!")
print(f"Device: {next(model.parameters()).device}")

## Cell 7: Load lại model sau training (nếu runtime bị reset)

## Cell 8: Tổng hợp kết quả

In [ ]:
print("="*60)
print("BẢNG KẾT QUẢ TỔNG HỢP")
print("="*60)
print(f"{'Phương pháp':<30} {'NLI Acc':>10} {'MCQ Acc':>10} {'Syl ROUGE-L':>12}")
print("-"*60)
print(f"{'Zero-shot':<30} {zs_nli_acc:>10.4f} {zs_mcq_acc:>10.4f} {zs_syl_rouge:>12.4f}")
print(f"{'QLoRA Multi-task (đề xuất)':<30} {ft_nli_acc:>10.4f} {ft_mcq_acc:>10.4f} {ft_syl_rouge:>12.4f}")
print("-"*60)

# Tính cải thiện
nli_imp = (ft_nli_acc - zs_nli_acc) / max(zs_nli_acc, 1e-6) * 100
mcq_imp = (ft_mcq_acc - zs_mcq_acc) / max(zs_mcq_acc, 1e-6) * 100
syl_imp = (ft_syl_rouge - zs_syl_rouge) / max(zs_syl_rouge, 1e-6) * 100

print(f"{'Cải thiện (%)':<30} {nli_imp:>+10.1f}% {mcq_imp:>+9.1f}% {syl_imp:>+11.1f}%")
print("="*60)

print("\n📋 COPY KẾT QUẢ NÀY VÀO BÁO CÁO:")
print(f"NLI: Zero-shot={zs_nli_acc:.4f}, QLoRA={ft_nli_acc:.4f}, Δ={nli_imp:+.1f}%")
print(f"MCQ: Zero-shot={zs_mcq_acc:.4f}, QLoRA={ft_mcq_acc:.4f}, Δ={mcq_imp:+.1f}%")
print(f"Syllogism ROUGE-L: Zero-shot={zs_syl_rouge:.4f}, QLoRA={ft_syl_rouge:.4f}, Δ={syl_imp:+.1f}%")
print(f"Training time: {training_time:.1f} phút")
print(f"Training loss: {train_result.training_loss:.4f}")

In [ ]:
# Lưu adapter
model.save_pretrained("./qwen_legal_lora_final")
tokenizer.save_pretrained("./qwen_legal_lora_final")
print("Đã lưu mô hình vào ./qwen_legal_lora_final")

# Tải về máy
import shutil
from google.colab import files
shutil.make_archive("qwen_legal_lora_final", 'zip', "./qwen_legal_lora_final")
files.download("qwen_legal_lora_final.zip")
print("Đang tải adapter về máy...")